# Notebook 03 — Feature Engineering

This notebook builds the key analysis variables from processed data:
1. **Heatwave detection** — region-specific thresholds, event identification
2. **Temperature features** — apparent temperature, WBGT, humidity
3. **RSVI construction** — percentile ranking, sub-indices, composite index
4. **Mortality rates** — age-standardized rates, excess mortality

**Prerequisites:** Run Notebook 02 first (interim data must exist).

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config, get_path
from src.utils.io import load_dataframe

cfg = load_config()
print("Ready.")

---
## 1. Heatwave Detection

### Definition
A **heatwave day** occurs when:
- Daily Tmax exceeds the **region-specific 90th percentile** of June–September Tmax, AND
- The exceedance lasts for **≥3 consecutive days**

### Steps
1. Compute the 90th percentile of summer Tmax for each region (2012–2022 climatology)
2. Flag days where Tmax exceeds the threshold
3. Identify consecutive runs of ≥3 days
4. Compute seasonal summary metrics (total HW days, events, intensity)

In [ ]:
# Load daily climate data
climate_path = get_path(cfg, 'interim_data') / 'daily_regional_climate.parquet'

if climate_path.exists():
    daily = pd.read_parquet(climate_path)
    daily['date'] = pd.to_datetime(daily['date'])
    print(f"Loaded {len(daily)} daily climate records")
    print(f"Regions: {daily['nuts2_code'].nunique()}")
    print(f"Date range: {daily['date'].min()} → {daily['date'].max()}")
else:
    print("❌ Daily climate data not found. Run Notebook 02 first.")
    print("   Using synthetic data for demonstration...")
    
    # Generate synthetic data for demonstration
    np.random.seed(42)
    dates = pd.date_range('2012-06-01', '2022-09-30', freq='D')
    dates = dates[dates.month.isin([6, 7, 8, 9])]
    regions = ['ITC4', 'ITF6', 'ITI4', 'ITF4', 'ITH4']
    records = []
    for region in regions:
        base = {'ITC4': 28, 'ITF6': 32, 'ITI4': 30, 'ITF4': 31, 'ITH4': 29}[region]
        for d in dates:
            records.append({
                'date': d, 'nuts2_code': region,
                'tmax': base + np.random.normal(0, 4),
                'tmin': base - 8 + np.random.normal(0, 3),
                'tmean': base - 4 + np.random.normal(0, 3),
                'dewpoint_mean': base - 15 + np.random.normal(0, 3),
            })
    daily = pd.DataFrame(records)
    print(f"Generated {len(daily)} synthetic records for demonstration")

In [ ]:
# Step 1: Compute region-specific percentile thresholds
from src.features.heatwave import compute_percentile_thresholds

hw_cfg = cfg['heatwave']
thresholds = compute_percentile_thresholds(
    daily,
    percentile=hw_cfg['percentile_threshold'],
    reference_start=hw_cfg['reference_start_year'],
    reference_end=hw_cfg['reference_end_year'],
)

print("\n90th Percentile Tmax Thresholds (°C):")
print(thresholds.sort_values(ascending=False).to_string())

In [ ]:
# Step 2: Detect heatwave days
from src.features.heatwave import detect_heatwave_days

hw_daily = detect_heatwave_days(
    daily, thresholds,
    min_consecutive=hw_cfg['min_consecutive_days'],
)

# Show results
n_hw = hw_daily['is_heatwave_day'].sum()
n_events = hw_daily['hw_event_id'].max()
print(f"\nTotal heatwave days: {n_hw}")
print(f"Total heatwave events: {n_events}")
print(f"\nHeatwave days per region:")
print(hw_daily.groupby('nuts2_code')['is_heatwave_day'].sum().sort_values(ascending=False))

In [ ]:
# Visualize: heatwave detection for one region in one year
region = hw_daily['nuts2_code'].unique()[0]
year = 2022

mask = (hw_daily['nuts2_code'] == region) & (hw_daily['date'].dt.year == year)
subset = hw_daily[mask].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(subset['date'], subset['tmax'], color='#2c3e50', linewidth=1, label='Daily Tmax')
ax.axhline(y=subset['threshold'].iloc[0], color='red', linestyle='--',
           alpha=0.7, label=f'90th pctl ({subset["threshold"].iloc[0]:.1f}°C)')

# Shade heatwave days
hw_mask = subset['is_heatwave_day']
ax.fill_between(subset['date'], ax.get_ylim()[0], ax.get_ylim()[1],
                where=hw_mask, alpha=0.2, color='red', label='Heatwave days')

ax.set_xlabel('Date')
ax.set_ylabel('Tmax (°C)')
ax.set_title(f'Heatwave Detection — {region}, Summer {year}', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step 3: Compute seasonal heatwave metrics
from src.features.heatwave import compute_heatwave_metrics

hw_metrics = compute_heatwave_metrics(hw_daily)

print(f"Heatwave metrics shape: {hw_metrics.shape}")
print(f"\nColumns: {hw_metrics.columns.tolist()}")
print(f"\nSample (top 10 by HW days):")
display(hw_metrics.nlargest(10, 'hw_days'))

---
## 2. Temperature Features (Humidity)

Compute derived temperature variables:
- **Relative humidity** (from T2m and dewpoint, Magnus formula)
- **Apparent temperature** (Steadman/NWS heat index)
- **WBGT approximation** (Wet-Bulb Globe Temperature)

In [ ]:
from src.features.temperature import (
    compute_relative_humidity,
    compute_apparent_temperature,
    compute_wbgt_approximation,
    add_temperature_features,
)

# Add derived features to daily climate data
daily_enhanced = add_temperature_features(daily)

print("New columns added:")
new_cols = ['rh_mean', 'apparent_tmax', 'wbgt_mean', 'wbgt_max']
for col in new_cols:
    if col in daily_enhanced.columns:
        print(f"  {col}: mean={daily_enhanced[col].mean():.1f}, "
              f"std={daily_enhanced[col].std():.1f}")

In [ ]:
# Visualize: Tmax vs Apparent Temperature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(daily_enhanced['tmax'], daily_enhanced['apparent_tmax'],
                alpha=0.1, s=5, c=daily_enhanced['rh_mean'], cmap='Blues')
axes[0].plot([20, 45], [20, 45], 'r--', alpha=0.5, label='1:1 line')
axes[0].set_xlabel('Tmax (°C)')
axes[0].set_ylabel('Apparent Tmax (°C)')
axes[0].set_title('Tmax vs Apparent Temperature')
axes[0].legend()

axes[1].hist(daily_enhanced['rh_mean'].dropna(), bins=50, color='steelblue', alpha=0.8)
axes[1].set_xlabel('Relative Humidity (%)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Mean Relative Humidity')

plt.suptitle('Humidity-Adjusted Temperature Features', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Regional Social Vulnerability Index (RSVI)

Following the CDC/ATSDR SVI methodology:

1. **Percentile-rank** each indicator across regions (within each year)
2. **Average into sub-indices** (Demographic, Economic, Urban)
3. **Composite RSVI** = mean of sub-indices → score between 0 and 1

Higher RSVI = higher social vulnerability.

In [ ]:
# Check if socioeconomic data exists
socio_path = get_path(cfg, 'interim_data') / 'socioeconomic_processed.parquet'

if socio_path.exists():
    socio = pd.read_parquet(socio_path)
    print(f"Loaded socioeconomic data: {socio.shape}")
    display(socio.head())
else:
    print("❌ Socioeconomic data not found.")
    print("   Using synthetic data for demonstration...")
    
    np.random.seed(42)
    regions = ['ITC4', 'ITF6', 'ITI4', 'ITF4', 'ITH4']
    years = range(2012, 2023)
    records = []
    for region in regions:
        # Create realistic variation by region
        vuln_base = {'ITC4': 0.3, 'ITF6': 0.8, 'ITI4': 0.5, 'ITF4': 0.7, 'ITH4': 0.4}[region]
        for year in years:
            records.append({
                'nuts2_code': region, 'year': year,
                'pct_pop_65plus': 20 + vuln_base * 10 + np.random.normal(0, 1),
                'pct_pop_75plus': 10 + vuln_base * 5 + np.random.normal(0, 0.5),
                'pct_pop_80plus': 5 + vuln_base * 3 + np.random.normal(0, 0.3),
                'poverty_rate_absolute': 5 + vuln_base * 20 + np.random.normal(0, 2),
                'gdp_per_capita_inv': -(40000 - vuln_base * 20000 + np.random.normal(0, 1000)),
                'disposable_income_inv': -(25000 - vuln_base * 10000 + np.random.normal(0, 500)),
                'population_density': 100 + vuln_base * 200 + np.random.normal(0, 20),
                'urbanization_rate': 50 + vuln_base * 30 + np.random.normal(0, 3),
            })
    socio = pd.DataFrame(records)
    print(f"Generated synthetic socioeconomic data: {socio.shape}")

In [ ]:
# Step 1: Percentile rank indicators
from src.features.rsvi import percentile_rank_within_year

rsvi_cfg = cfg['rsvi']
all_indicators = []
for indicators in rsvi_cfg['sub_indices'].values():
    all_indicators.extend(indicators)

print(f"Indicators to rank: {all_indicators}")
ranked = percentile_rank_within_year(socio, all_indicators)

# Show percentile columns
pctl_cols = [c for c in ranked.columns if c.endswith('_pctl')]
print(f"\nPercentile columns created: {pctl_cols}")
display(ranked[['nuts2_code', 'year'] + pctl_cols].head(10))

In [ ]:
# Step 2: Compute sub-indices
from src.features.rsvi import compute_sub_indices

with_subs = compute_sub_indices(ranked, rsvi_cfg['sub_indices'])

sub_cols = [c for c in with_subs.columns if c.startswith('subidx_')]
print(f"Sub-indices: {sub_cols}")
display(with_subs[['nuts2_code', 'year'] + sub_cols].head(10))

In [ ]:
# Step 3: Composite RSVI
from src.features.rsvi import compute_composite_rsvi

sub_names = list(rsvi_cfg['sub_indices'].keys())
rsvi_df = compute_composite_rsvi(with_subs, sub_names, method=rsvi_cfg['aggregation'])

print("\nRSVI by region (mean across years):")
rsvi_summary = rsvi_df.groupby('nuts2_code')['rsvi'].mean().sort_values(ascending=False)
print(rsvi_summary.to_string())

In [ ]:
# Visualize RSVI components
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of sub-indices by region
pivot = rsvi_df.groupby('nuts2_code')[sub_cols + ['rsvi']].mean()
pivot = pivot.sort_values('rsvi', ascending=False)
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('RSVI Components by Region', fontweight='bold')

# RSVI distribution
axes[1].hist(rsvi_df['rsvi'], bins=20, color='#e74c3c', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('RSVI Score')
axes[1].set_ylabel('Count')
axes[1].set_title('RSVI Distribution', fontweight='bold')
axes[1].axvline(rsvi_df['rsvi'].median(), color='black', linestyle='--', label='Median')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4. Save Feature Outputs

Save all engineered features to the interim directory.

In [ ]:
from src.utils.io import save_dataframe

interim_dir = get_path(cfg, 'interim_data')

# Save heatwave metrics
save_dataframe(hw_metrics, interim_dir / 'heatwave_metrics.parquet', index=False)

# Save RSVI
rsvi_output = rsvi_df[['nuts2_code', 'year', 'rsvi'] + sub_cols].copy()
save_dataframe(rsvi_output, interim_dir / 'rsvi.parquet', index=False)

print("\n✅ Features saved to data/interim/")
print("\n➡️ Next: Notebook 04 — Panel Assembly & EDA")